In [15]:
import numpy as np

np.random.seed(42)

X = np.random.uniform(-2, 2, (400, 3))
y = (np.sin(X[:, 0]) + 0.5 * (X[:, 1]**2) - 0.8 * X[:, 2]).reshape(-1, 1)

X = X.T
y = y.T

print("Dataset generated. Shapes -> X:", X.shape, "y:", y.shape)

Dataset generated. Shapes -> X: (3, 400) y: (1, 400)


In [16]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)

def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_derivative(z):
    return sigmoid(z)

In [17]:
def initialize_parameters(layer_dims):
    parameters = {}
    for l in range(1, len(layer_dims)):
        parameters['W' + str(l)] = np.random.uniform(
            -0.5, 0.5, (layer_dims[l], layer_dims[l-1])
        )
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

def forward_propagation(X, parameters, activation_func):
    caches = []
    A = X
    L = len(parameters) // 2

    for l in range(1, L):
        A_prev = A
        W = parameters['W' + str(l)]
        b = parameters['b' + str(l)]
        Z = W @ A_prev + b
        A = activation_func(Z)
        caches.append((A_prev, W, b, Z))

    A_prev = A
    W = parameters['W' + str(L)]
    b = parameters['b' + str(L)]
    Z = W @ A_prev + b
    A = Z
    caches.append((A_prev, W, b, Z))

    return A, caches

def backward_propagation(y_hat, y, caches, activation_derivative):
    gradients = {}
    L = len(caches)
    m = y.shape[1]

    dA = 2 * (y_hat - y)

    for l in reversed(range(L)):
        A_prev, W, b, Z = caches[l]

        if l == L - 1:
            dZ = dA
        else:
            dZ = dA * activation_derivative(Z)

        gradients['dW' + str(l+1)] = (1/m) * (dZ @ A_prev.T)
        gradients['db' + str(l+1)] = (1/m) * np.sum(dZ, axis=1, keepdims=True)
        dA = W.T @ dZ

    return gradients

def update_parameters(parameters, gradients, learning_rate):
    L = len(parameters) // 2
    for l in range(1, L + 1):
        parameters['W' + str(l)] -= learning_rate * gradients['dW' + str(l)]
        parameters['b' + str(l)] -= learning_rate * gradients['db' + str(l)]
    return parameters

In [18]:
def train_model(layer_dims, activation_func, activation_deriv, epochs=1000, lr=0.01):
    parameters = initialize_parameters(layer_dims)

    for epoch in range(epochs):
        y_hat, caches = forward_propagation(X, parameters, activation_func)

        loss = (1 / y.shape[1]) * np.sum((y_hat - y) ** 2)

        gradients = backward_propagation(y_hat, y, caches, activation_deriv)
        parameters = update_parameters(parameters, gradients, lr)

        if epoch == 199:
            loss_200 = loss

    grad_norm_first = np.linalg.norm(gradients['dW1'])
    grad_norm_last = np.linalg.norm(
        gradients['dW' + str(len(layer_dims) - 2)]
    )

    return loss, loss_200, grad_norm_first, grad_norm_last

In [19]:
print(f"{'Model':<22} | {'Act.':<8} | {'Final Loss':<12} | {'Loss @ 200':<12} | {'Grad Norm L1':<14} | {'Grad Norm Last':<14}")
print("-" * 100)

models = {
    "Model A (Shallow)": ([3, 4, 1], relu, relu_derivative),
    "Model B (Medium)": ([3, 6, 6, 1], relu, relu_derivative),
    "Model C (Deep)": ([3, 8, 8, 8, 8, 1], relu, relu_derivative),
    "Model D (Very Deep) - ReLU": ([3, 8, 8, 8, 8, 8, 8, 8, 8, 1], relu, relu_derivative),
    "Model D (Very Deep) - Sigmoid": ([3, 8, 8, 8, 8, 8, 8, 8, 8, 1], sigmoid, sigmoid_derivative)
}

for name, (dims, act, deriv) in models.items():
    act_name = "ReLU" if act == relu else "Sigmoid"
    final_loss, loss_200, norm_first, norm_last = train_model(dims, act, deriv)

    print(f"{name:<22} | {act_name:<8} | {final_loss:<12.4f} | {loss_200:<12.4f} | {norm_first:<14.6f} | {norm_last:<14.6f}")

Model                  | Act.     | Final Loss   | Loss @ 200   | Grad Norm L1   | Grad Norm Last
----------------------------------------------------------------------------------------------------
Model A (Shallow)      | ReLU     | 0.1115       | 0.4938       | 0.045217       | 0.045217      
Model B (Medium)       | ReLU     | 0.0729       | 0.3220       | 0.036609       | 0.021441      
Model C (Deep)         | ReLU     | 0.0305       | 0.8620       | 0.023876       | 0.016801      
Model D (Very Deep) - ReLU | ReLU     | 0.0532       | 1.6349       | 0.429784       | 0.621290      
Model D (Very Deep) - Sigmoid | Sigmoid  | 1.7439       | 1.7439       | 0.000006       | 0.000006      
